In [ ]:
import torch 
import pandas as pd 
import sys 
sys.path.append('../../python')
from ddms.surrogate import Trainer
from ddms.surrogate.dataset import MemoryDataset_LSTM
from ddms.surrogate.model import LMSC
from ddms.surrogate.tensor import *
from ddms.surrogate.mechanics import *
from ddms.surrogate.processing import *

## torchscript modules

In [ ]:
dataset = MemoryDataset_LSTM
model = LMSC

args = Args(
    mode='torchscript',
    exp_name='LMSC',
    run_name='script_data',
    run_id='vvna6z5m',
    data_path='', 
    device='cpu',
)
trainer = Trainer(args)
trainer.set_dataset(dataset)
trainer.set_model(model)

script_data = torch.jit.script(ScriptData(data_scale=trainer.scaler.data_scale))
script_model = torch.jit.script(trainer.model)
torch.jit.save(script_data, trainer.tracker.get_artifact_uri('script_data.pt'))
torch.jit.save(script_model, trainer.tracker.get_artifact_uri('script_model.pt'))

In [ ]:
trainer.scaler.data_scale

## create abaqus amplitude 
- work with `macro_modify_amplitude.py` (in `abaqus/`)

In [ ]:
# for cube single element 

# SEcomplex
# result_path = r''
# save_path = r''

# SEtensionx
# result_path = r''
# save_path = r''

# SEshearxy
result_path = r''
save_path = r''

pts = 8     # cube 
size = 10   # length
edge = torch.tensor([0, size])
X0 = torch.meshgrid(edge, edge, edge)
X0 = torch.stack(X0).reshape(3, -1).t().double()        # (pt, 3)
X = get_amplitude(X0, result_path)                      # (pt, s, 3), coordinates
U = X.permute(1,0,2) - X0                               # (s, pt, 3), displacement
U = U.reshape(-1, pts*3)                                # (s, pt*3)

# add time zero
U = torch.cat([U.new_zeros(1, U.size(1)), U], dim=0)    # (s+1, pt*3)

# add max 
idx = torch.argmax(U.abs(), dim=0, keepdim=True)
U_max = U.gather(dim=0, index=idx)
U = torch.cat([U_max, U/U_max], dim=0)                  # (s+2, pt*3)
U = torch.nan_to_num(U, nan=0)

# add time index
index = ['max']
index += [t for t in range(0,1011,10)]

df = pd.DataFrame(U, index=index)
df.to_csv(save_path)